# Attack Detection and Probability Estimation

## 1. Modeling Objective

This notebook focuses on developing machine learning models for binary network traffic classification using the preprocessed CICIoMT2024 dataset.

The primary objective is to distinguish benign network traffic from malicious traffic:

- `0`: Benign
- `1`: Attack

In addition to classification performance, predicted attack probabilities will be examined to support subsequent risk assessment and prioritization tasks.

Model development will follow a leakage-aware workflow. Feature scaling, class imbalance handling, cross-validation, and probability calibration will be performed using only the training data, while the independent test set will be reserved for final evaluation.

## 2. Data Loading

The modeling-ready datasets generated during the preprocessing stage are stored in Parquet format.

At this stage, only the training dataset is loaded for model development. The independent test dataset is intentionally kept separate and will not be accessed until the final evaluation stage.

This separation helps prevent unintended information leakage and preserves the independence of the final test set.

### 2.1. Training Dataset Loading

The preprocessed training dataset contains the 42 selected input features and the binary target variable (`Label`).

The dataset is loaded directly from the previously exported Parquet file, eliminating the need to repeat the preprocessing and feature selection procedures performed in the previous notebook.

In [1]:
from pathlib import Path
import pandas as pd

processed_dir = Path("data/processed")

train_path = processed_dir / "train_modeling.parquet"
test_path = processed_dir / "test_modeling.parquet"

df_train = pd.read_parquet(train_path)

print("Training dataset shape:", df_train.shape)
print("Training columns:", len(df_train.columns))
print("Target column available:", "Label" in df_train.columns)

Training dataset shape: (7160831, 43)
Training columns: 43
Target column available: True


### 2.1. Findings

The modeling-ready training dataset was successfully loaded from Parquet format.

The dataset contains 7,160,831 observations and 43 columns, consisting of 42 selected input features and the binary target variable (`Label`).

The target variable was confirmed to be available, and no additional preprocessing was repeated during data loading.

## 3. Feature and Target Separation

The binary target variable (`Label`) is separated conceptually from the input features before model development.

To reduce unnecessary memory usage, the complete feature matrix is not duplicated at this stage. Instead, the feature column names are defined and will be used to access the required observations during model development.

The independent test dataset remains untouched.

In [2]:
feature_columns = [
    column for column in df_train.columns
    if column != "Label"
]

y_train = df_train["Label"]

print("Training records:", len(df_train))
print("Feature count:", len(feature_columns))
print("Target shape:", y_train.shape)
print("Target classes:", sorted(y_train.unique()))
print("Label included in features:", "Label" in feature_columns)

Training records: 7160831
Feature count: 42
Target shape: (7160831,)
Target classes: [np.int64(0), np.int64(1)]
Label included in features: False


### 3. Findings

The training dataset contains 7,160,831 observations and 42 input features.

The binary target variable (`Label`) was successfully separated from the feature set and contains the expected classes:

- `0`: Benign
- `1`: Attack

The target variable is not included among the model input features.

No additional copy of the full feature matrix was created, reducing unnecessary memory usage.

## 4. Multicollinearity and Feature Redundancy Assessment

Highly correlated features identified during exploratory analysis are reassessed before model development.

A representative random sample of the training dataset is used to reduce computational and memory requirements while preserving the original training distribution.

Feature pairs with an absolute Pearson correlation coefficient greater than 0.90 are identified as potential redundancy candidates.

High correlation alone will not be used as an automatic criterion for feature removal. Final decisions will also consider feature meaning and model performance.

## 4.1. High-Correlation Feature Pairs

In [3]:
correlation_sample = df_train[
    feature_columns
].sample(
    n=500_000,
    random_state=42
)

correlation_matrix = correlation_sample.corr()

high_correlation_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) > 0.90:
            high_correlation_pairs.append({
                "Feature_1": correlation_matrix.columns[i],
                "Feature_2": correlation_matrix.columns[j],
                "Correlation": correlation
            })

high_correlation_df = (
    pd.DataFrame(high_correlation_pairs)
    .sort_values(
        by="Correlation",
        key=abs,
        ascending=False
    )
    .reset_index(drop=True)
)

print("Sample size:", len(correlation_sample))
print("High-correlation pairs:", len(high_correlation_df))

display(high_correlation_df)

Sample size: 500000
High-correlation pairs: 15


,Feature_1,Feature_2,Correlation
0,ARP,IPv,-1.000000
1,Radius,Std,0.999972
2,Number,Weight,0.999736
3,IAT,Weight,0.998571
4,IAT,Number,0.997726
5,AVG,Tot size,0.978524
6,AVG,Magnitue,0.974522
7,Covariance,Radius,0.960617
8,Covariance,Std,0.960606
9,Magnitue,Tot size,0.954618


### 4.1. Findings

The correlation reassessment identified 15 feature pairs with an absolute Pearson correlation coefficient greater than 0.90.

Several pairs exhibited extremely strong relationships, including `ARP`–`IPv`, `Radius`–`Std`, `Number`–`Weight`, `IAT`–`Weight`, and `IAT`–`Number`.

The overall correlation structure is consistent with the exploratory analysis performed in the previous notebook, although the exact coefficients differ slightly because the current analysis uses a new random sample of the training dataset.

These highly correlated features represent potential redundancy candidates. However, no additional features are removed at this stage, since correlation alone is insufficient to justify feature elimination.

The impact of correlated features will be evaluated during model development, particularly for models that are sensitive to multicollinearity.

### 4.2. Correlated Feature Group Analysis

Highly correlated feature pairs are grouped into connected redundancy clusters.

This grouping provides a clearer view of features that may contain overlapping information and helps define candidate feature subsets for subsequent model comparison.

No features are removed during this stage.

In [4]:
correlation_groups = []

remaining_features = set(
    high_correlation_df["Feature_1"]
).union(
    high_correlation_df["Feature_2"]
)

while remaining_features:
    current_group = {remaining_features.pop()}

    changed = True

    while changed:
        changed = False

        for _, row in high_correlation_df.iterrows():
            feature_1 = row["Feature_1"]
            feature_2 = row["Feature_2"]

            if feature_1 in current_group and feature_2 not in current_group:
                current_group.add(feature_2)
                remaining_features.discard(feature_2)
                changed = True

            elif feature_2 in current_group and feature_1 not in current_group:
                current_group.add(feature_1)
                remaining_features.discard(feature_1)
                changed = True

    correlation_groups.append(sorted(current_group))

for index, group in enumerate(correlation_groups, start=1):
    print(f"Group {index}: {group}")

Group 1: ['AVG', 'Magnitue', 'Max', 'Tot size', 'Tot sum']
Group 2: ['Protocol Type', 'UDP']
Group 3: ['Covariance', 'Radius', 'Std']
Group 4: ['IAT', 'Number', 'Weight']
Group 5: ['ARP', 'IPv']


### 4.2. Findings

The highly correlated features were organized into five redundancy groups:

1. `IAT`, `Number`, and `Weight`
2. `AVG`, `Magnitue`, `Max`, `Tot size`, and `Tot sum`
3. `Protocol Type` and `UDP`
4. `Covariance`, `Radius`, and `Std`
5. `ARP` and `IPv`

These groups indicate sets of features that may contain overlapping information.

No features were removed at this stage. The identified groups will be used to define reduced feature configurations and compare their effect on model performance during subsequent experiments.

## 5. Development Dataset Preparation

A stratified development subset is created from the full training dataset to support efficient model development and comparison.

The subset preserves the original binary class distribution while reducing computational and memory requirements.

This subset will be used for preliminary model experiments and model selection. The full training dataset remains unchanged and will be used later for final model training.

### 5.1. Stratified Development Sampling

A stratified sample of 300,000 observations is selected from the full training dataset.

Stratification ensures that the proportions of benign and attack observations remain consistent with the original training distribution.

In [5]:
import gc
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

# Release temporary objects used during correlation analysis
del correlation_sample
del correlation_matrix
gc.collect()

development_size = 300_000

splitter = StratifiedShuffleSplit(
    n_splits=1,
    train_size=development_size,
    random_state=42
)

placeholder = np.empty((len(df_train), 0))

dev_indices, remaining_indices = next(
    splitter.split(placeholder, y_train)
)

del placeholder
del remaining_indices
gc.collect()

X_dev = df_train.iloc[dev_indices][feature_columns]
y_dev = df_train.iloc[dev_indices]["Label"]

print("Development features:", X_dev.shape)
print("Development target:", y_dev.shape)

print("\nClass distribution:")
print(y_dev.value_counts().sort_index())

print("\nClass percentages:")
print(
    (y_dev.value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

Development features: (300000, 42)
Development target: (300000,)

Class distribution:
Label
0      8074
1    291926
Name: count, dtype: int64

Class percentages:
Label
0     2.69
1    97.31
Name: proportion, dtype: float64


### 5.1. Findings

A stratified development subset containing 300,000 observations and 42 input features was successfully created.

The subset contains 8,074 benign observations (2.69%) and 291,926 attack observations (97.31%).

The original class distribution of the full training dataset was preserved, confirming that the stratified sampling procedure was successful.

The development subset will be used for preliminary model experiments and comparison, while the full training dataset remains unchanged for later final model training.

### 5.2. Development Train–Validation Split

The development dataset is divided into training and validation subsets using stratified sampling.

An 80:20 split is applied while preserving the original class distribution in both subsets.

The training subset will be used for model fitting, while the validation subset will support preliminary model comparison and selection.

In [6]:
from sklearn.model_selection import train_test_split

X_dev_train, X_dev_val, y_dev_train, y_dev_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    stratify=y_dev,
    random_state=42
)

print("Development training features:", X_dev_train.shape)
print("Development validation features:", X_dev_val.shape)

print("Development training target:", y_dev_train.shape)
print("Development validation target:", y_dev_val.shape)

print("\nTraining class percentages:")
print(
    (y_dev_train.value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

print("\nValidation class percentages:")
print(
    (y_dev_val.value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

Development training features: (240000, 42)
Development validation features: (60000, 42)
Development training target: (240000,)
Development validation target: (60000,)

Training class percentages:
Label
0     2.69
1    97.31
Name: proportion, dtype: float64

Validation class percentages:
Label
0     2.69
1    97.31
Name: proportion, dtype: float64


### 5.2. Findings

The development dataset was successfully divided into training and validation subsets using an 80:20 stratified split.

The development training set contains 240,000 observations, while the validation set contains 60,000 observations.

The original class distribution was preserved in both subsets, with approximately 2.69% benign traffic and 97.31% attack traffic.

The validation subset will be used for preliminary model comparison and selection, while the independent test dataset remains untouched for final evaluation.

## 6. Baseline Model Development

A baseline classification model is established before evaluating machine learning algorithms.

Because the training data is highly imbalanced, a majority-class classifier is used to quantify the performance that can be achieved without learning meaningful relationships between the input features and the target variable.

This baseline provides a reference point for evaluating subsequent models.

### 6.1. Majority-Class Baseline

A `DummyClassifier` using the most frequent class strategy is trained on the development training set.

The model always predicts the majority class and therefore represents a non-informative baseline.

Accuracy, balanced accuracy, macro F1-score, classification metrics, and the confusion matrix are evaluated on the development validation set.

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_dev_train,
    y_dev_train
)

y_dummy_pred = dummy_model.predict(X_dev_val)

print("Accuracy:",
      f"{accuracy_score(y_dev_val, y_dummy_pred):.4f}")

print("Balanced Accuracy:",
      f"{balanced_accuracy_score(y_dev_val, y_dummy_pred):.4f}")

print("Macro F1:",
      f"{f1_score(y_dev_val, y_dummy_pred, average='macro'):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_dummy_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_dev_val, y_dummy_pred))

Accuracy: 0.9731
Balanced Accuracy: 0.5000
Macro F1: 0.4932

Classification Report:
              precision    recall  f1-score   support

      Benign     0.0000    0.0000    0.0000      1615
      Attack     0.9731    1.0000    0.9864     58385

    accuracy                         0.9731     60000
   macro avg     0.4865    0.5000    0.4932     60000
weighted avg     0.9469    0.9731    0.9598     60000

Confusion Matrix:
[[    0  1615]
 [    0 58385]]


### 6.1. Findings

The majority-class baseline achieved an accuracy of 97.31%, despite failing to identify any benign observations.

All 1,615 benign validation records were incorrectly classified as attacks, resulting in a benign recall and F1-score of 0.0000.

In contrast, all 58,385 attack observations were correctly classified, producing an attack recall of 1.0000.

The balanced accuracy of 0.5000 and macro F1-score of 0.4932 demonstrate that the high overall accuracy is primarily caused by the severe class imbalance rather than meaningful predictive performance.

These results confirm that accuracy alone is not an appropriate evaluation metric for this dataset. Subsequent models will therefore be evaluated using class-specific precision, recall, F1-score, balanced accuracy, and other imbalance-sensitive metrics.

## 7. Logistic Regression

### 7.1. Logistic Regression Classifier

Logistic Regression is evaluated as an additional linear classification model for distinguishing benign and malicious network traffic.

Because the model is sensitive to feature scale, feature standardization is incorporated into the preprocessing pipeline.

Class weighting is applied to account for the severe imbalance between benign and attack observations.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

logistic_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

logistic_pipeline.fit(
    X_dev_train,
    y_dev_train
)

y_logistic_pred = logistic_pipeline.predict(X_dev_val)

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_logistic_pred):.4f}"
)

print(
    "Balanced Accuracy:",
    f"{balanced_accuracy_score(y_dev_val, y_logistic_pred):.4f}"
)

print(
    "Macro F1:",
    f"{f1_score(y_dev_val, y_logistic_pred, average='macro'):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_logistic_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_logistic_pred
    )
)

Accuracy: 0.9887
Balanced Accuracy: 0.9924
Macro F1: 0.9098

Classification Report:
              precision    recall  f1-score   support

      Benign     0.7048    0.9963    0.8256      1615
      Attack     0.9999    0.9885    0.9941     58385

    accuracy                         0.9887     60000
   macro avg     0.8523    0.9924    0.9098     60000
weighted avg     0.9920    0.9887    0.9896     60000

Confusion Matrix:
[[ 1609     6]
 [  674 57711]]


### 7. Findings

The Logistic Regression baseline substantially outperformed the majority-class classifier.

The model achieved an accuracy of 98.87%, a balanced accuracy of 99.24%, and a macro F1-score of 0.9098.

Benign traffic was identified with a recall of 99.63%, with 1,609 of 1,615 benign observations correctly classified. However, benign precision was lower at 70.48%, indicating that some attack observations were incorrectly classified as benign.

For the attack class, the model achieved a precision of 99.99% and a recall of 98.85%.

The confusion matrix showed 6 benign observations incorrectly classified as attacks and 674 attack observations incorrectly classified as benign.

These results demonstrate that class weighting and feature scaling enable Logistic Regression to handle the severe class imbalance effectively, although false negatives in the attack class remain an important consideration.

## 8. Random Forest

### 8.1. Random Forest Classifier

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_dev_train,
    y_dev_train
)

y_rf_pred = random_forest_model.predict(X_dev_val)

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_rf_pred):.4f}"
)

print(
    "Balanced Accuracy:",
    f"{balanced_accuracy_score(y_dev_val, y_rf_pred):.4f}"
)

print(
    "Macro F1:",
    f"{f1_score(y_dev_val, y_rf_pred, average='macro'):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_rf_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_rf_pred
    )
)

Accuracy: 0.9991
Balanced Accuracy: 0.9914
Macro F1: 0.9911

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9821    0.9833    0.9827      1615
      Attack     0.9995    0.9995    0.9995     58385

    accuracy                         0.9991     60000
   macro avg     0.9908    0.9914    0.9911     60000
weighted avg     0.9991    0.9991    0.9991     60000

Confusion Matrix:
[[ 1588    27]
 [   29 58356]]


### 8.1. Findings

The Random Forest classifier achieved very strong performance on the development validation set, with an accuracy of 99.91%, balanced accuracy of 99.14%, and macro F1-score of 0.9911.

For benign traffic, the model achieved a precision of 98.21%, recall of 98.33%, and F1-score of 0.9827.

For attack traffic, precision and recall both reached 99.95%, resulting in an F1-score of 0.9995.

The confusion matrix showed that 27 benign observations were incorrectly classified as attacks, while 29 attack observations were incorrectly classified as benign.

Overall, Random Forest demonstrated highly balanced performance across both classes and provided particularly strong attack detection capability.

In [13]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

model_predictions = {
    "Logistic Regression": y_logistic_pred,
    "Random Forest": y_rf_pred,
    "HistGradientBoosting": y_hist_pred
}

comparison_results = []

for model_name, predictions in model_predictions.items():
    comparison_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_dev_val,
            predictions
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_dev_val,
            predictions
        ),
        "Macro F1": f1_score(
            y_dev_val,
            predictions,
            average="macro"
        ),
        "Benign Recall": recall_score(
            y_dev_val,
            predictions,
            pos_label=0
        ),
        "Attack Recall": recall_score(
            y_dev_val,
            predictions,
            pos_label=1
        )
    })

model_comparison = (
    pd.DataFrame(comparison_results)
    .sort_values(
        by="Macro F1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    model_comparison.style.format({
        "Accuracy": "{:.4f}",
        "Balanced Accuracy": "{:.4f}",
        "Macro F1": "{:.4f}",
        "Benign Recall": "{:.4f}",
        "Attack Recall": "{:.4f}"
    })
)

,Model,Accuracy,Balanced Accuracy,Macro F1,Benign Recall,Attack Recall
0,Random Forest,0.9991,0.9914,0.9911,0.9833,0.9995
1,HistGradientBoosting,0.9987,0.9963,0.9875,0.9938,0.9988
2,Logistic Regression,0.9887,0.9924,0.9098,0.9963,0.9885


## 9. Feature Redundancy Experiment

A reduced feature configuration is evaluated to determine whether near-redundant features can be removed without degrading predictive performance.

Only extremely correlated relationships with an absolute Pearson correlation coefficient of at least 0.99 are considered at this stage. Features with lower correlations are retained to avoid overly aggressive feature elimination.

The reduced configuration will be compared with the original 42-feature configuration using the same development training and validation subsets.

### 9.1. Reduced Feature Set Construction

A conservative reduced feature set is constructed from the strongest redundancy groups.

For each near-redundant group, an interpretable representative feature is retained while highly correlated alternatives are excluded.

The resulting feature configuration is experimental and does not modify the original 42-feature dataset.

In [14]:
redundant_candidates = [
    "IPv",
    "Radius",
    "Number",
    "Weight"
]

reduced_feature_columns = [
    feature
    for feature in feature_columns
    if feature not in redundant_candidates
]

X_dev_train_reduced = X_dev_train[
    reduced_feature_columns
]

X_dev_val_reduced = X_dev_val[
    reduced_feature_columns
]

print("Original feature count:", len(feature_columns))
print("Reduced feature count:", len(reduced_feature_columns))
print("Removed candidate features:", redundant_candidates)

print(
    "Reduced training shape:",
    X_dev_train_reduced.shape
)

print(
    "Reduced validation shape:",
    X_dev_val_reduced.shape
)

Original feature count: 42
Reduced feature count: 38
Removed candidate features: ['IPv', 'Radius', 'Number', 'Weight']
Reduced training shape: (240000, 38)
Reduced validation shape: (60000, 38)


### 9.1. Findings

A reduced feature set containing 38 input features was successfully constructed.

Four highly correlated candidate features (`IPv`, `Radius`, `Number`, and `Weight`) were excluded from the experimental configuration.

The original 42-feature dataset remains unchanged. The reduced feature set will be evaluated against the original configuration to determine whether feature redundancy can be reduced without sacrificing predictive performance.

## 9.2. Reduced Feature Model Evaluation

The Random Forest classifier is retrained using the reduced 38-feature configuration.

The same development training and validation observations are used to ensure a fair comparison with the original 42-feature model.

The objective is to determine whether removing near-redundant features affects predictive performance.

In [20]:
random_forest_reduced = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

random_forest_reduced.fit(
    X_dev_train_reduced,
    y_dev_train
)

y_rf_reduced_pred = random_forest_reduced.predict(
    X_dev_val_reduced
)

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_rf_reduced_pred):.4f}"
)

print(
    "Balanced Accuracy:",
    f"{balanced_accuracy_score(y_dev_val, y_rf_reduced_pred):.4f}"
)

print(
    "Macro F1:",
    f"{f1_score(y_dev_val, y_rf_reduced_pred, average='macro'):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_rf_reduced_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_rf_reduced_pred
    )
)

Accuracy: 0.9991
Balanced Accuracy: 0.9911
Macro F1: 0.9911

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9827    0.9827    0.9827      1615
      Attack     0.9995    0.9995    0.9995     58385

    accuracy                         0.9991     60000
   macro avg     0.9911    0.9911    0.9911     60000
weighted avg     0.9991    0.9991    0.9991     60000

Confusion Matrix:
[[ 1587    28]
 [   28 58357]]


### 9.2. Findings

The Random Forest classifier maintained nearly identical performance after reducing the feature set from 42 to 38 features.

Accuracy remained unchanged at 99.91%, while macro F1-score also remained constant at 0.9911.

Balanced accuracy decreased only marginally from 0.9914 to 0.9911.

The reduced model produced 28 false-positive and 28 false-negative predictions, compared with 27 false positives and 29 false negatives in the original 42-feature configuration.

Overall, removing the four near-redundant features did not result in a meaningful loss of predictive performance.

These results support retaining the 38-feature configuration for subsequent model development.

## 9.3. Class-Specific Performance Comparison

The original 42-feature and reduced 38-feature Random Forest models are compared using class-specific recall and confusion matrix components.

This comparison focuses on whether feature reduction affects the model's ability to correctly identify benign and attack observations.

Particular attention is given to false-negative attack predictions, as undetected attacks represent a critical error in intrusion detection systems.

In [21]:
from sklearn.metrics import recall_score, confusion_matrix

rf_comparison_results = []

rf_models = {
    "Random Forest - 42 Features": y_rf_pred,
    "Random Forest - 38 Features": y_rf_reduced_pred
}

for model_name, predictions in rf_models.items():
    tn, fp, fn, tp = confusion_matrix(
        y_dev_val,
        predictions
    ).ravel()

    rf_comparison_results.append({
        "Model": model_name,
        "Benign Recall": recall_score(
            y_dev_val,
            predictions,
            pos_label=0
        ),
        "Attack Recall": recall_score(
            y_dev_val,
            predictions,
            pos_label=1
        ),
        "False Positives": fp,
        "False Negatives": fn
    })

rf_feature_comparison = pd.DataFrame(
    rf_comparison_results
)

display(
    rf_feature_comparison.style.format({
        "Benign Recall": "{:.4f}",
        "Attack Recall": "{:.4f}"
    })
)

,Model,Benign Recall,Attack Recall,False Positives,False Negatives
0,Random Forest - 42 Features,0.9833,0.9995,27,29
1,Random Forest - 38 Features,0.9827,0.9995,28,28


### 9.3. Findings

The class-specific comparison showed that reducing the feature set from 42 to 38 features had a negligible effect on Random Forest performance.

Benign recall decreased slightly from 0.9833 to 0.9827, while attack recall remained unchanged at 0.9995.

The 42-feature model produced 27 false positives and 29 false negatives, whereas the 38-feature model produced 28 false positives and 28 false negatives.

The reduced feature configuration therefore preserved the model's attack detection capability while slightly decreasing feature redundancy.

Based on these results, the 38-feature configuration will be retained for subsequent model development and comparison.

## 10. XGBoost

### 10.1. XGBoost Classifier

XGBoost is evaluated as a gradient-boosted decision tree model for binary network attack detection.

The model is trained using the reduced 38-feature configuration and the same development training and validation subsets used for the previous classifiers.

Because the dataset is highly imbalanced, balanced sample weights are incorporated during model training.

The model is evaluated using accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix.

In [26]:
# %pip install xgboost

In [27]:
from xgboost import XGBClassifier

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

xgb_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_dev_train
)

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_dev_train_reduced,
    y_dev_train,
    sample_weight=xgb_sample_weights
)

y_xgb_pred = xgb_model.predict(
    X_dev_val_reduced
)

y_xgb_proba = xgb_model.predict_proba(
    X_dev_val_reduced
)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_xgb_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_dev_val, y_xgb_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_dev_val, y_xgb_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_dev_val, y_xgb_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_dev_val, y_xgb_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_xgb_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_xgb_pred
    )
)

Accuracy: 0.9990
Precision: 0.9998
Recall: 0.9991
F1-score: 0.9995
ROC-AUC: 1.0000

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9686    0.9944    0.9814      1615
      Attack     0.9998    0.9991    0.9995     58385

    accuracy                         0.9990     60000
   macro avg     0.9842    0.9968    0.9904     60000
weighted avg     0.9990    0.9990    0.9990     60000

Confusion Matrix:
[[ 1606     9]
 [   52 58333]]


### 10.1. Findings

The XGBoost classifier achieved an accuracy of 99.90%, precision of 99.98%, recall of 99.91%, and F1-score of 0.9995 for the attack class.

The model also achieved a ROC-AUC score of 1.0000, indicating extremely strong separation between benign and attack observations on the development validation set.

For benign traffic, the model achieved a recall of 99.44%, correctly identifying 1,606 of 1,615 benign observations.

The confusion matrix showed 9 benign observations incorrectly classified as attacks and 52 attack observations incorrectly classified as benign.

Overall, XGBoost demonstrated highly balanced predictive performance. However, the near-perfect ROC-AUC score should be validated carefully using the independent test dataset before drawing conclusions about final generalization performance.

## 11. LightGBM

### 11.1. LightGBM Classifier

LightGBM is evaluated as a gradient boosting framework for binary network attack detection.

The model is trained using the reduced 38-feature configuration and the same development training and validation subsets used for the previous classifiers.

Class weighting is applied to address the severe imbalance between benign and attack observations.

Performance is evaluated using accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix.

In [31]:
# %pip install lightgbm

In [32]:
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

lightgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model.fit(
    X_dev_train_reduced,
    y_dev_train
)

y_lgbm_pred = lightgbm_model.predict(
    X_dev_val_reduced
)

y_lgbm_proba = lightgbm_model.predict_proba(
    X_dev_val_reduced
)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_lgbm_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_dev_val, y_lgbm_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_dev_val, y_lgbm_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_dev_val, y_lgbm_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_dev_val, y_lgbm_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_lgbm_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_lgbm_pred
    )
)

Accuracy: 0.9990
Precision: 0.9998
Recall: 0.9991
F1-score: 0.9995
ROC-AUC: 0.9999

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9692    0.9938    0.9814      1615
      Attack     0.9998    0.9991    0.9995     58385

    accuracy                         0.9990     60000
   macro avg     0.9845    0.9965    0.9904     60000
weighted avg     0.9990    0.9990    0.9990     60000

Confusion Matrix:
[[ 1605    10]
 [   51 58334]]


### 11.1. Findings

The LightGBM classifier achieved an accuracy of 99.90%, precision of 99.98%, recall of 99.91%, and F1-score of 0.9995 for the attack class.

The model achieved a ROC-AUC score of 0.9999, indicating excellent discrimination between benign and malicious network traffic.

For benign traffic, the model achieved a recall of 99.38%, correctly identifying 1,605 of 1,615 benign observations.

The confusion matrix showed 10 benign observations incorrectly classified as attacks and 51 attack observations incorrectly classified as benign.

Overall, LightGBM demonstrated performance very similar to XGBoost, with strong results across both classes. Final model selection will therefore require direct comparison of all candidate models under the same evaluation framework.

## 12. Multilayer Perceptron

### 12.1. MLP Classifier

A Multilayer Perceptron is evaluated as a neural network-based approach for binary network attack detection.

Because neural networks are sensitive to differences in feature magnitude, StandardScaler is incorporated into the preprocessing pipeline.

The model is trained using the reduced 38-feature configuration and the same development training and validation subsets used for the previous classifiers.

Performance is evaluated using accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix.

In [34]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

mlp_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        MLPClassifier(
            hidden_layer_sizes=(100, 50),
            activation="relu",
            solver="adam",
            learning_rate_init=0.001,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=42
        )
    )
])

mlp_model.fit(
    X_dev_train_reduced,
    y_dev_train
)

y_mlp_pred = mlp_model.predict(
    X_dev_val_reduced
)

y_mlp_proba = mlp_model.predict_proba(
    X_dev_val_reduced
)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_mlp_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_dev_val, y_mlp_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_dev_val, y_mlp_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_dev_val, y_mlp_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_dev_val, y_mlp_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_mlp_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_mlp_pred
    )
)

Accuracy: 0.9970
Precision: 0.9993
Recall: 0.9977
F1-score: 0.9985
ROC-AUC: 0.9997

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9215    0.9734    0.9467      1615
      Attack     0.9993    0.9977    0.9985     58385

    accuracy                         0.9970     60000
   macro avg     0.9604    0.9855    0.9726     60000
weighted avg     0.9972    0.9970    0.9971     60000

Confusion Matrix:
[[ 1572    43]
 [  134 58251]]


### 12.1. Findings

The Multilayer Perceptron achieved an accuracy of 99.70%, precision of 99.93%, recall of 99.77%, and F1-score of 0.9985 for the attack class.

The model also achieved a ROC-AUC score of 0.9997, indicating strong discrimination between benign and malicious traffic.

For benign traffic, the model achieved a recall of 97.34%, correctly identifying 1,572 of 1,615 benign observations.

The confusion matrix showed 43 benign observations incorrectly classified as attacks and 134 attack observations incorrectly classified as benign.

Although the MLP demonstrated strong overall predictive performance, its class-specific results were weaker than those obtained by the tree-based ensemble models, particularly in terms of benign recall and false-negative attack predictions.

# 13. Model Comparison

## 13.1. Comparative Model Evaluation

The five classification models are compared using the same 38-feature development training and validation datasets.

The evaluated models are:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM
- Multilayer Perceptron

Model performance is assessed using the evaluation metrics specified for the study:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

For precision, recall, and F1-score, the attack class (`1`) is treated as the positive class.

ROC-AUC is calculated using predicted attack probabilities rather than binary class predictions.

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Logistic Regression with the final 38-feature configuration
logistic_model_reduced = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

logistic_model_reduced.fit(
    X_dev_train_reduced,
    y_dev_train
)

y_logistic_reduced_pred = logistic_model_reduced.predict(
    X_dev_val_reduced
)

y_logistic_reduced_proba = logistic_model_reduced.predict_proba(
    X_dev_val_reduced
)[:, 1]

# Probability predictions for models already trained
y_rf_reduced_proba = random_forest_reduced.predict_proba(
    X_dev_val_reduced
)[:, 1]

model_outputs = {
    "Logistic Regression": (
        y_logistic_reduced_pred,
        y_logistic_reduced_proba
    ),
    "Random Forest": (
        y_rf_reduced_pred,
        y_rf_reduced_proba
    ),
    "XGBoost": (
        y_xgb_pred,
        y_xgb_proba
    ),
    "LightGBM": (
        y_lgbm_pred,
        y_lgbm_proba
    ),
    "MLP": (
        y_mlp_pred,
        y_mlp_proba
    )
}

comparison_results = []

for model_name, (predictions, probabilities) in model_outputs.items():
    comparison_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_dev_val,
            predictions
        ),
        "Precision": precision_score(
            y_dev_val,
            predictions
        ),
        "Recall": recall_score(
            y_dev_val,
            predictions
        ),
        "F1-score": f1_score(
            y_dev_val,
            predictions
        ),
        "ROC-AUC": roc_auc_score(
            y_dev_val,
            probabilities
        )
    })

model_comparison = (
    pd.DataFrame(comparison_results)
    .sort_values(
        by="F1-score",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    model_comparison.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Random Forest,0.9991,0.9995,0.9995,0.9995,0.9999
1,LightGBM,0.9990,0.9998,0.9991,0.9995,0.9999
2,XGBoost,0.9990,0.9998,0.9991,0.9995,1.0000
3,MLP,0.9970,0.9993,0.9977,0.9985,0.9997
4,Logistic Regression,0.9886,0.9999,0.9884,0.9941,0.9979


### 13.1. Findings

The comparative evaluation demonstrated that the tree-based ensemble models achieved the strongest overall performance.

Random Forest obtained the highest accuracy (0.9991) and recall (0.9995), while maintaining a precision and F1-score of 0.9995.

XGBoost and LightGBM produced nearly identical classification performance, achieving an accuracy of 0.9990, precision of 0.9998, recall of 0.9991, and F1-score of 0.9995.

XGBoost achieved the highest ROC-AUC score (1.0000), followed closely by Random Forest and LightGBM (0.9999). The near-perfect ROC-AUC values indicate very strong class separability on the development validation set and will require confirmation on the independent test dataset.

The MLP classifier also demonstrated strong performance but remained below the tree-based ensemble models.

Logistic Regression produced the lowest accuracy, recall, F1-score, and ROC-AUC among the evaluated learning models, despite achieving very high attack precision.

Based on the development validation results, Random Forest, XGBoost, and LightGBM are retained as the primary candidates for hyperparameter optimization.

### 13.1.1. Baseline Model Results Persistence

The validation performance of the initial candidate models is stored for subsequent comparison with the hyperparameter-optimized models.

In [59]:
baseline_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "LightGBM",
        "XGBoost",
        "MLP",
        "Logistic Regression"
    ],
    "Accuracy": [
        0.9991,
        0.9990,
        0.9990,
        0.9970,
        0.9886
    ],
    "Precision": [
        0.9995,
        0.9998,
        0.9998,
        0.9993,
        0.9999
    ],
    "Recall": [
        0.9995,
        0.9991,
        0.9991,
        0.9977,
        0.9884
    ],
    "F1-score": [
        0.9995,
        0.9995,
        0.9995,
        0.9985,
        0.9941
    ],
    "ROC-AUC": [
        0.9999,
        0.9999,
        1.0000,
        0.9997,
        0.9979
    ],
    "False Negative": [
        confusion_matrix(y_dev_val, y_rf_pred)[1, 0],
        confusion_matrix(y_dev_val, y_lgbm_pred)[1, 0],
        confusion_matrix(y_dev_val, y_xgb_pred)[1, 0],
        confusion_matrix(y_dev_val, y_mlp_pred)[1, 0],
        confusion_matrix(y_dev_val, y_logistic_reduced_pred)[1, 0]
    ]
})

baseline_results.to_csv(
    result_dir / "baseline_model_results.csv",
    index=False
)

print("Baseline model results saved successfully.")

Baseline model results saved successfully.


## 14. Hyperparameter Optimization

Hyperparameter optimization is performed for the strongest candidate models identified during the initial model comparison.

The optimization procedure uses only the development training subset. The development validation set remains excluded from the search process and will be used later to compare the optimized models.

A stratified cross-validation strategy is applied to preserve the class distribution across folds.

Because the dataset is highly imbalanced, macro F1-score is used as the optimization criterion so that both benign and attack classes contribute equally to model selection.

### 14.1. Random Forest Hyperparameter Optimization

Randomized hyperparameter search is applied to the Random Forest classifier.

A limited search space is used to balance computational cost and model optimization quality.

The search evaluates combinations of tree count, tree depth, feature sampling, minimum split size, minimum leaf size, and class weighting using stratified cross-validation.

In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)
from sklearn.metrics import make_scorer, fbeta_score


rf_tuning_model = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

rf_param_distributions = {
    "n_estimators": [190,200,210],
    "max_depth": [19,20,21],
    "min_samples_split": [9,10,11],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt", "log2", 0.5],
    "class_weight": [
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 1},
        {0: 1, 1: 2},
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 2, 1: 1},
        {0: 5, 1: 1},
        {0: 10, 1: 1},
    ]
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    pos_label=1
)

rf_random_search = RandomizedSearchCV(
    estimator=rf_tuning_model,
    param_distributions=rf_param_distributions,
    n_iter=30,
    scoring=f2_scorer,
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

rf_random_search.fit(
    X_dev_train_reduced,
    y_dev_train
)

print("Best Parameters:")
print(rf_random_search.best_params_)

print(
    "\nBest Cross-Validation F2-score:",
    f"{rf_random_search.best_score_:.4f}"
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'n_estimators': 190, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_depth': 19, 'class_weight': {0: 1, 1: 5}}

Best Cross-Validation F2-score: 0.9995


### 14.1. Findings

Randomized hyperparameter optimization was performed using five-fold stratified cross-validation and F2-score as the optimization criterion.

F2-score was selected because false-negative attack predictions are considered more critical than false-positive predictions in the intrusion detection context.

The best-performing Random Forest configuration was:

- `n_estimators = 190`
- `max_depth = 19`
- `min_samples_split = 10`
- `min_samples_leaf = 3`
- `max_features = sqrt`
- `class_weight = {0: 1, 1: 5}`

The optimized configuration achieved a mean cross-validation F2-score of 0.9995.

The selected class weighting assigns greater importance to the attack class, which is consistent with the objective of reducing missed attack detections.

## 14.2. Tuned Random Forest Validation Evaluation

The optimized Random Forest model is evaluated on the development validation set.

Its performance is compared with the original Random Forest configuration to determine whether hyperparameter optimization provides a meaningful improvement.

The validation set was not used during the hyperparameter search.

In [46]:
rf_tuned_model = rf_random_search.best_estimator_

y_rf_tuned_pred = rf_tuned_model.predict(
    X_dev_val_reduced
)

y_rf_tuned_proba = rf_tuned_model.predict_proba(
    X_dev_val_reduced
)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_rf_tuned_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_dev_val, y_rf_tuned_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_dev_val, y_rf_tuned_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_dev_val, y_rf_tuned_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_dev_val, y_rf_tuned_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_rf_tuned_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_rf_tuned_pred
    )
)

Accuracy: 0.9989
Precision: 0.9992
Recall: 0.9997
F1-score: 0.9994
ROC-AUC: 0.9999

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9880    0.9703    0.9791      1615
      Attack     0.9992    0.9997    0.9994     58385

    accuracy                         0.9989     60000
   macro avg     0.9936    0.9850    0.9892     60000
weighted avg     0.9989    0.9989    0.9989     60000

Confusion Matrix:
[[ 1567    48]
 [   19 58366]]


## 14.3. XGBoost Hyperparameter Optimization

Hyperparameter optimization is performed for the XGBoost classifier using five-fold stratified cross-validation.

F2-score is used as the optimization criterion because reducing false-negative attack predictions is considered more important than minimizing false-positive predictions.

Since the attack class is represented by `1`, `scale_pos_weight` is included in the search space to investigate whether assigning additional importance to attack observations improves detection recall.

In [47]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb_tuning_model = XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

xgb_param_distributions = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.05, 0.10, 0.15],
    "max_depth": [4, 6, 8],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.1, 0.5],
    "scale_pos_weight": [1, 2, 5, 10]
}

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_tuning_model,
    param_distributions=xgb_param_distributions,
    n_iter=30,
    scoring=f2_scorer,
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

xgb_random_search.fit(
    X_dev_train_reduced,
    y_dev_train
)

print("Best Parameters:")
print(xgb_random_search.best_params_)

print(
    "\nBest Cross-Validation F2-score:",
    f"{xgb_random_search.best_score_:.4f}"
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'subsample': 1.0, 'scale_pos_weight': 5, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.5, 'colsample_bytree': 0.8}

Best Cross-Validation F2-score: 0.9994


### 14.3. Findings

The XGBoost hyperparameter search achieved a mean five-fold cross-validation F2-score of 0.9994.

The best-performing configuration was:

- `n_estimators = 200`
- `learning_rate = 0.1`
- `max_depth = 8`
- `min_child_weight = 3`
- `gamma = 0.5`
- `subsample = 1.0`
- `colsample_bytree = 0.8`
- `scale_pos_weight = 5`

The selected `scale_pos_weight` assigns greater importance to the attack class during model training, which is consistent with the objective of reducing false-negative attack predictions.

The high cross-validation F2-score indicates strong recall-oriented performance while maintaining high precision across the stratified folds.

## 14.4. Tuned XGBoost Validation Evaluation

The F2-optimized XGBoost model is evaluated on the development validation set.

The evaluation focuses particularly on attack recall and false-negative predictions to determine whether hyperparameter optimization improves the detection of malicious traffic.

In [48]:
xgb_tuned_model = xgb_random_search.best_estimator_

y_xgb_tuned_pred = xgb_tuned_model.predict(
    X_dev_val_reduced
)

y_xgb_tuned_proba = xgb_tuned_model.predict_proba(
    X_dev_val_reduced
)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_xgb_tuned_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_dev_val, y_xgb_tuned_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_dev_val, y_xgb_tuned_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_dev_val, y_xgb_tuned_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_dev_val, y_xgb_tuned_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_xgb_tuned_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_xgb_tuned_pred
    )
)

Accuracy: 0.9987
Precision: 0.9991
Recall: 0.9995
F1-score: 0.9993
ROC-AUC: 1.0000

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9830    0.9690    0.9760      1615
      Attack     0.9991    0.9995    0.9993     58385

    accuracy                         0.9987     60000
   macro avg     0.9911    0.9843    0.9877     60000
weighted avg     0.9987    0.9987    0.9987     60000

Confusion Matrix:
[[ 1565    50]
 [   27 58358]]


### 14.4. Findings

The F2-optimized XGBoost model achieved an accuracy of 0.9987, precision of 0.9991, recall of 0.9995, F1-score of 0.9993, and ROC-AUC of 1.0000 on the development validation set.

Compared with the original XGBoost configuration, the number of false-negative attack predictions decreased substantially from 52 to 27, resulting in improved attack recall.

However, the number of false-positive predictions increased from 9 to 50, accompanied by small reductions in accuracy, precision, and F1-score.

This behavior is consistent with the F2-based optimization objective, which prioritizes recall and therefore favors configurations that reduce missed attack detections.

Overall, the tuned XGBoost model provides a more recall-oriented operating point that is better aligned with the objective of minimizing undetected attacks.

## 14.5. LightGBM Hyperparameter Optimization

Hyperparameter optimization is performed for the LightGBM classifier using five-fold stratified cross-validation.

F2-score is used as the optimization criterion to prioritize attack recall and reduce false-negative attack predictions.

For binary classification, `scale_pos_weight` is included in the search space to evaluate different levels of importance assigned to the attack class.

In [51]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV


lgbm_tuning_model = LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=1,
    verbosity=-1,
    subsample_freq=1
)

lgbm_param_distributions = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.05, 0.10, 0.15],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 10, 20],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0],
    "scale_pos_weight": [1, 2, 5, 10]
}

lgbm_random_search = RandomizedSearchCV(
    estimator=lgbm_tuning_model,
    param_distributions=lgbm_param_distributions,
    n_iter=30,
    scoring=f2_scorer,
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

lgbm_random_search.fit(
    X_dev_train_reduced,
    y_dev_train
)

print("Best Parameters:")
print(lgbm_random_search.best_params_)

print(
    "\nBest Cross-Validation F2-score:",
    f"{lgbm_random_search.best_score_:.4f}"
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'subsample': 1.0, 'scale_pos_weight': 10, 'reg_lambda': 0.0, 'reg_alpha': 1.0, 'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 30, 'max_depth': 20, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Best Cross-Validation F2-score: 0.9994


### 14.5. Findings

The LightGBM hyperparameter search achieved a mean five-fold cross-validation F2-score of 0.9994.

The best-performing configuration was:

- `n_estimators = 200`
- `learning_rate = 0.05`
- `num_leaves = 31`
- `max_depth = 20`
- `min_child_samples = 30`
- `subsample = 1.0`
- `colsample_bytree = 0.8`
- `reg_alpha = 1.0`
- `reg_lambda = 0.0`
- `scale_pos_weight = 10`

The selected `scale_pos_weight` assigns substantially greater importance to the attack class, which is consistent with the objective of minimizing false-negative attack predictions.

The mean cross-validation F2-score of 0.9994 indicates strong recall-oriented performance across the stratified folds.

## 14.6. Tuned LightGBM Validation Evaluation

The F2-optimized LightGBM model is evaluated on the development validation set.

The evaluation focuses particularly on attack recall and false-negative predictions to determine whether hyperparameter optimization improves the detection of malicious traffic.

In [52]:
lgbm_tuned_model = lgbm_random_search.best_estimator_

y_lgbm_tuned_pred = lgbm_tuned_model.predict(
    X_dev_val_reduced
)

y_lgbm_tuned_proba = lgbm_tuned_model.predict_proba(
    X_dev_val_reduced
)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_dev_val, y_lgbm_tuned_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_dev_val, y_lgbm_tuned_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_dev_val, y_lgbm_tuned_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_dev_val, y_lgbm_tuned_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_dev_val, y_lgbm_tuned_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_dev_val,
        y_lgbm_tuned_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_dev_val,
        y_lgbm_tuned_pred
    )
)

Accuracy: 0.9986
Precision: 0.9989
Recall: 0.9997
F1-score: 0.9993
ROC-AUC: 1.0000

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9873    0.9610    0.9740      1615
      Attack     0.9989    0.9997    0.9993     58385

    accuracy                         0.9986     60000
   macro avg     0.9931    0.9803    0.9866     60000
weighted avg     0.9986    0.9986    0.9986     60000

Confusion Matrix:
[[ 1552    63]
 [   20 58365]]


### 14.6. Findings

The F2-optimized LightGBM model achieved an accuracy of 0.9986, precision of 0.9989, recall of 0.9997, F1-score of 0.9993, and ROC-AUC of 1.0000 on the development validation set.

Compared with the original LightGBM configuration, the number of false-negative attack predictions decreased substantially from 51 to 20, resulting in improved attack recall.

However, the number of false-positive predictions increased from 10 to 63, accompanied by small reductions in accuracy, precision, and F1-score.

This trade-off is consistent with the F2-based optimization objective, which prioritizes recall and reduces missed attack detections at the cost of additional false-positive predictions.

Overall, the tuned LightGBM model provides a recall-oriented operating point that is more aligned with the objective of minimizing undetected attacks.

### 14.7. Tuned Model Persistence

The optimized Random Forest, XGBoost, and LightGBM models, together with their best hyperparameters and cross-validation results, are stored for subsequent comparison and final model selection.

In [53]:
from pathlib import Path
import joblib
import json
import pandas as pd

model_dir = Path("models")
result_dir = Path("results")

model_dir.mkdir(exist_ok=True)
result_dir.mkdir(exist_ok=True)


# Random Forest
joblib.dump(
    rf_random_search.best_estimator_,
    model_dir / "random_forest_f2_tuned.joblib"
)

with open(
    result_dir / "random_forest_f2_best_params.json",
    "w"
) as file:
    json.dump(
        rf_random_search.best_params_,
        file,
        indent=4
    )

pd.DataFrame(
    rf_random_search.cv_results_
).to_csv(
    result_dir / "random_forest_f2_cv_results.csv",
    index=False
)


# XGBoost
joblib.dump(
    xgb_random_search.best_estimator_,
    model_dir / "xgboost_f2_tuned.joblib"
)

with open(
    result_dir / "xgboost_f2_best_params.json",
    "w"
) as file:
    json.dump(
        xgb_random_search.best_params_,
        file,
        indent=4
    )

pd.DataFrame(
    xgb_random_search.cv_results_
).to_csv(
    result_dir / "xgboost_f2_cv_results.csv",
    index=False
)


# LightGBM
joblib.dump(
    lgbm_random_search.best_estimator_,
    model_dir / "lightgbm_f2_tuned.joblib"
)

with open(
    result_dir / "lightgbm_f2_best_params.json",
    "w"
) as file:
    json.dump(
        lgbm_random_search.best_params_,
        file,
        indent=4
    )

pd.DataFrame(
    lgbm_random_search.cv_results_
).to_csv(
    result_dir / "lightgbm_f2_cv_results.csv",
    index=False
)

print("All tuned models and tuning results saved successfully.")

All tuned models and tuning results saved successfully.


## 14.7. Tuned Model Comparison

The validation performances of the F2-optimized Random Forest, XGBoost, and LightGBM models are compared using the same evaluation metrics.

Since the optimization objective prioritizes missed attack reduction, false-negative counts are also included in the comparison.

In [60]:
tuned_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "Accuracy": [
        accuracy_score(y_dev_val, y_rf_tuned_pred),
        accuracy_score(y_dev_val, y_xgb_tuned_pred),
        accuracy_score(y_dev_val, y_lgbm_tuned_pred)
    ],
    "Precision": [
        precision_score(y_dev_val, y_rf_tuned_pred),
        precision_score(y_dev_val, y_xgb_tuned_pred),
        precision_score(y_dev_val, y_lgbm_tuned_pred)
    ],
    "Recall": [
        recall_score(y_dev_val, y_rf_tuned_pred),
        recall_score(y_dev_val, y_xgb_tuned_pred),
        recall_score(y_dev_val, y_lgbm_tuned_pred)
    ],
    "F1-score": [
        f1_score(y_dev_val, y_rf_tuned_pred),
        f1_score(y_dev_val, y_xgb_tuned_pred),
        f1_score(y_dev_val, y_lgbm_tuned_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_dev_val, y_rf_tuned_proba),
        roc_auc_score(y_dev_val, y_xgb_tuned_proba),
        roc_auc_score(y_dev_val, y_lgbm_tuned_proba)
    ],
    "False Negative": [
        confusion_matrix(y_dev_val, y_rf_tuned_pred)[1, 0],
        confusion_matrix(y_dev_val, y_xgb_tuned_pred)[1, 0],
        confusion_matrix(y_dev_val, y_lgbm_tuned_pred)[1, 0]
    ]
})

tuned_results = tuned_results.sort_values(
    by=["False Negative", "Recall"],
    ascending=[True, False]
).reset_index(drop=True)

display(
    tuned_results.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)

tuned_results.to_csv(
    result_dir / "tuned_model_results.csv",
    index=False
)

print("Tuned model comparison results saved successfully.")

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,False Negative
0,Random Forest,0.9989,0.9992,0.9997,0.9994,0.9999,19
1,LightGBM,0.9986,0.9989,0.9997,0.9993,1.0000,20
2,XGBoost,0.9987,0.9991,0.9995,0.9993,1.0000,27


Tuned model comparison results saved successfully.


### 15. Findings

The optimized Random Forest achieved the lowest number of false-negative attack predictions, with only 19 missed attacks on the development validation set.

LightGBM produced a comparable attack recall but resulted in 20 false-negative predictions, while XGBoost missed 27 attack observations.

Although LightGBM and XGBoost achieved a slightly higher ROC-AUC value of 1.0000, the Random Forest provided the most favorable balance between attack recall, precision, F1-score, and false-negative reduction.

Given that minimizing undetected attacks is the primary optimization objective, the F2-optimized Random Forest is selected as the leading candidate for final model training and independent test evaluation.

# 16. Final Model Selection

Based on the optimized model comparison, the F2-optimized Random Forest is selected as the final attack detection model.

The model achieved the lowest number of false-negative predictions, missing only 19 attack observations on the development validation set. It also maintained an accuracy of 0.9989, precision of 0.9992, recall of 0.9997, F1-score of 0.9994, and ROC-AUC of 0.9999.

Although LightGBM achieved a comparable recall and a slightly higher rounded ROC-AUC value, it produced 20 false-negative predictions. XGBoost resulted in 27 false-negative predictions.

Since minimizing undetected attacks is considered the primary objective of the detection system, the optimized Random Forest provides the most appropriate trade-off between recall, precision, overall classification performance, and false-negative reduction.

Therefore, the optimized Random Forest configuration will be retrained using the complete training dataset before final evaluation on the independent test set.

# 17. Full Training Dataset Training

Following final model selection, the F2-optimized Random Forest configuration is retrained using the complete training dataset.

The final model uses the 38-feature reduced representation selected during the feature redundancy analysis.

The independent test dataset remains completely excluded from this stage and will only be used for the final performance evaluation.

In [61]:
import gc

variables_to_remove = [
    "X_dev_train",
    "X_dev_val",
    "X_dev_train_reduced",
    "X_dev_val_reduced",
    "y_dev_train",
    "y_dev_val"
]

for variable in variables_to_remove:
    if variable in globals():
        del globals()[variable]

gc.collect()

print("Development datasets released from memory.")

Development datasets released from memory.


In [62]:
from sklearn.ensemble import RandomForestClassifier

final_rf_model = RandomForestClassifier(
    n_estimators=190,
    max_depth=19,
    min_samples_split=10,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight={0: 1, 1: 5},
    random_state=42,
    n_jobs=-1
)

print("Training records:", len(df_train))
print("Number of features:", len(reduced_feature_columns))

final_rf_model.fit(
    df_train[reduced_feature_columns],
    df_train["Label"]
)

print("Final Random Forest training completed successfully.")

Training records: 7160831
Number of features: 38
Final Random Forest training completed successfully.


## 17.1. Final Model Persistence

The final Random Forest model trained on the complete training dataset is stored together with the selected feature list to ensure reproducibility and consistent feature ordering during independent test evaluation and future deployment.

In [64]:
import joblib
import json

joblib.dump(
    final_rf_model,
    model_dir / "final_attack_detection_model.joblib"
)

with open(
    model_dir / "final_feature_columns.json",
    "w"
) as file:
    json.dump(
        reduced_feature_columns,
        file,
        indent=4
    )

print("Final model and feature list saved successfully.")

Final model and feature list saved successfully.


# 18. Independent Test Evaluation

The final Random Forest model is evaluated on the independent test dataset that remained completely excluded from model development, hyperparameter optimization, and model selection.

This evaluation provides the final estimate of the model's generalization performance on previously unseen data.

In [65]:
import gc
import pandas as pd

if "df_train" in globals():
    del df_train

gc.collect()

test_path = processed_dir / "test_modeling.parquet"

df_test = pd.read_parquet(test_path)

print("Independent test records:", len(df_test))
print("Test columns:", df_test.shape[1])
print("Selected features available:",
      all(feature in df_test.columns for feature in reduced_feature_columns))
print("Target available:", "Label" in df_test.columns)
print("Missing values:",
      df_test[reduced_feature_columns + ["Label"]].isna().sum().sum())

Independent test records: 1613825
Test columns: 43
Selected features available: True
Target available: True
Missing values: 0


## 18.1. Final Test Prediction and Evaluation

The final Random Forest model is evaluated on the independent test dataset using the selected 38-feature representation.

Performance is assessed using Accuracy, Precision, Recall, F1-score, ROC-AUC, and the confusion matrix. Particular attention is given to false-negative attack predictions because missed attacks represent the primary operational risk.

In [66]:
X_test = df_test[reduced_feature_columns]
y_test = df_test["Label"]

y_test_pred = final_rf_model.predict(X_test)

y_test_proba = final_rf_model.predict_proba(X_test)[:, 1]

print(
    "Accuracy:",
    f"{accuracy_score(y_test, y_test_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_test, y_test_pred):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_test, y_test_pred):.4f}"
)

print(
    "F1-score:",
    f"{f1_score(y_test, y_test_pred):.4f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, y_test_proba):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Benign", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_test_pred
    )
)

Accuracy: 0.9982
Precision: 0.9984
Recall: 0.9998
F1-score: 0.9991
ROC-AUC: 0.9999

Classification Report:
              precision    recall  f1-score   support

      Benign     0.9893    0.9321    0.9599     37607
      Attack     0.9984    0.9998    0.9991   1576218

    accuracy                         0.9982   1613825
   macro avg     0.9939    0.9659    0.9795   1613825
weighted avg     0.9982    0.9982    0.9982   1613825

Confusion Matrix:
[[  35053    2554]
 [    378 1575840]]


### 18.1. Findings

The final Random Forest model achieved an accuracy of 0.9982, precision of 0.9984, recall of 0.9998, F1-score of 0.9991, and ROC-AUC of 0.9999 on the independent test dataset.

Out of 1,576,218 attack observations, only 378 were incorrectly classified as benign, resulting in an attack recall of 0.9998. This indicates that the model successfully detected nearly all malicious traffic in the independent test set.

The model produced 2,554 false-positive predictions, where benign observations were classified as attacks. Consequently, benign recall decreased to 0.9321.

This behavior is consistent with the F2-oriented hyperparameter optimization strategy, which prioritized minimizing false-negative attack predictions over reducing false-positive alerts.

Despite the class imbalance and the substantially larger independent test set, the model maintained strong generalization performance with high precision, recall, F1-score, and ROC-AUC.

Overall, the final Random Forest model demonstrates strong effectiveness for binary attack detection, particularly in minimizing undetected malicious traffic.

## 18.2. Training Performance Assessment

The final model is also evaluated on the complete training dataset to examine the difference between training and independent test performance.

These results are not considered an unbiased estimate of generalization performance because the model was trained on the same observations.

In [67]:
df_train_final = pd.read_parquet(
    processed_dir / "train_modeling.parquet",
    columns=reduced_feature_columns + ["Label"]
)

X_train_final = df_train_final[reduced_feature_columns]
y_train_final = df_train_final["Label"]

y_train_final_pred = final_rf_model.predict(X_train_final)
y_train_final_proba = final_rf_model.predict_proba(X_train_final)[:, 1]

train_final_metrics = {
    "Accuracy": accuracy_score(y_train_final, y_train_final_pred),
    "Precision": precision_score(y_train_final, y_train_final_pred),
    "Recall": recall_score(y_train_final, y_train_final_pred),
    "F1-score": f1_score(y_train_final, y_train_final_pred),
    "ROC-AUC": roc_auc_score(y_train_final, y_train_final_proba),
    "False Negative": confusion_matrix(
        y_train_final,
        y_train_final_pred
    )[1, 0]
}

for metric, value in train_final_metrics.items():
    if metric == "False Negative":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

Accuracy: 0.9993
Precision: 0.9993
Recall: 0.9999
F1-score: 0.9996
ROC-AUC: 1.0000
False Negative: 412


In [68]:
final_performance = pd.DataFrame([
    {
        "Dataset": "Training",
        **train_final_metrics
    },
    {
        "Dataset": "Independent Test",
        "Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision": precision_score(y_test, y_test_pred),
        "Recall": recall_score(y_test, y_test_pred),
        "F1-score": f1_score(y_test, y_test_pred),
        "ROC-AUC": roc_auc_score(y_test, y_test_proba),
        "False Negative": confusion_matrix(y_test, y_test_pred)[1, 0]
    }
])

display(
    final_performance.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)

final_performance.to_csv(
    result_dir / "final_model_performance.csv",
    index=False
)

,Dataset,Accuracy,Precision,Recall,F1-score,ROC-AUC,False Negative
0,Training,0.9993,0.9993,0.9999,0.9996,1.0000,412
1,Independent Test,0.9982,0.9984,0.9998,0.9991,0.9999,378


# 19. Attack Probability Analysis

The final Random Forest model provides probabilistic estimates representing the likelihood that each observation belongs to the attack class.

These probabilities are analyzed to examine the model's confidence and to better understand the separation between benign and malicious traffic.

## 19.1. Attack Probability Distribution

In [69]:
probability_analysis = pd.DataFrame({
    "True_Label": y_test.values,
    "Attack_Probability": y_test_proba
})

print("Overall Attack Probability Summary:")
print(
    probability_analysis["Attack_Probability"].describe()
)

print("\nMean Attack Probability by True Class:")
print(
    probability_analysis.groupby("True_Label")[
        "Attack_Probability"
    ].mean()
)

print("\nMedian Attack Probability by True Class:")
print(
    probability_analysis.groupby("True_Label")[
        "Attack_Probability"
    ].median()
)

Overall Attack Probability Summary:
count    1.613825e+06
mean     9.780803e-01
std      1.420830e-01
min      0.000000e+00
25%      9.999999e-01
50%      9.999999e-01
75%      9.999999e-01
max      1.000000e+00
Name: Attack_Probability, dtype: float64

Mean Attack Probability by True Class:
True_Label
0    0.082208
1    0.999455
Name: Attack_Probability, dtype: float64

Median Attack Probability by True Class:
True_Label
0    0.002917
1    1.000000
Name: Attack_Probability, dtype: float64


### 19.1. Findings

The predicted attack probabilities demonstrate a strong separation between benign and malicious observations.

Attack observations received a mean attack probability of 0.9995 and a median probability of 1.0000, indicating that the model assigns very high confidence to the majority of malicious traffic.

In contrast, benign observations received a substantially lower mean attack probability of 0.0822 and a median probability of only 0.0029.

The difference between the benign mean and median suggests that most benign observations receive very low attack probabilities, while a relatively small subset receives considerably higher probabilities.

The high overall mean attack probability of 0.9781 primarily reflects the strong class imbalance in the independent test dataset, where attack observations constitute the large majority of records.

## 19.2. Probability Analysis of Misclassified Observations

The predicted probabilities of false-positive and false-negative observations are examined to determine the confidence associated with the model's classification errors.

In [70]:
probability_analysis["Predicted_Label"] = y_test_pred

false_positives = probability_analysis[
    (probability_analysis["True_Label"] == 0) &
    (probability_analysis["Predicted_Label"] == 1)
]

false_negatives = probability_analysis[
    (probability_analysis["True_Label"] == 1) &
    (probability_analysis["Predicted_Label"] == 0)
]

print("False Positive Count:", len(false_positives))
print("False Negative Count:", len(false_negatives))

print("\nFalse Positive Attack Probability Summary:")
print(
    false_positives["Attack_Probability"].describe()
)

print("\nFalse Negative Attack Probability Summary:")
print(
    false_negatives["Attack_Probability"].describe()
)

False Positive Count: 2554
False Negative Count: 378

False Positive Attack Probability Summary:
count    2554.000000
mean        0.715848
std         0.100973
min         0.500031
25%         0.638311
50%         0.734640
75%         0.794013
max         0.911691
Name: Attack_Probability, dtype: float64

False Negative Attack Probability Summary:
count    378.000000
mean       0.371280
std        0.098808
min        0.017348
25%        0.305907
50%        0.396254
75%        0.448130
max        0.499756
Name: Attack_Probability, dtype: float64


### 19.2. Findings

The probability analysis of misclassified observations shows that most classification errors occur relatively close to the default decision threshold of 0.5.

False-positive observations had a mean attack probability of 0.7158 and a median of 0.7346, indicating that the model generally assigned moderate-to-high attack confidence to benign observations that were incorrectly classified as attacks.

False-negative observations had a mean attack probability of 0.3713 and a median of 0.3963. Their maximum attack probability reached 0.4998, demonstrating that several missed attacks were located very close to the classification threshold.

These results indicate that some false-negative predictions may potentially be recovered by lowering the decision threshold below 0.5.

However, such a threshold adjustment would likely increase the number of false-positive alerts and should therefore be evaluated as a recall–precision trade-off.

## 19.3. Decision Threshold Analysis

Different decision thresholds are evaluated to examine the trade-off between false-negative and false-positive predictions.

Since missed attacks are considered more critical than false alarms, lower classification thresholds are investigated to determine whether attack recall can be improved without causing an excessive increase in false-positive predictions.

In [71]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

threshold_results = []

for threshold in thresholds:
    y_threshold_pred = (
        y_test_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_threshold_pred
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_test,
            y_threshold_pred
        ),
        "Precision": precision_score(
            y_test,
            y_threshold_pred
        ),
        "Recall": recall_score(
            y_test,
            y_threshold_pred
        ),
        "F1-score": f1_score(
            y_test,
            y_threshold_pred
        ),
        "False Positive": fp,
        "False Negative": fn
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(
    threshold_results.style.format({
        "Threshold": "{:.2f}",
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}"
    })
)

,Threshold,Accuracy,Precision,Recall,F1-score,False Positive,False Negative
0,0.50,0.9982,0.9984,0.9998,0.9991,2554,378
1,0.45,0.9981,0.9982,0.9998,0.9990,2826,285
2,0.40,0.9979,0.9980,0.9999,0.9989,3137,195
3,0.35,0.9978,0.9978,0.9999,0.9989,3460,131
4,0.30,0.9976,0.9976,0.9999,0.9988,3766,90


### 19.3. Findings

Lowering the classification threshold progressively reduced the number of false-negative attack predictions while increasing the number of false-positive alerts.

At the default threshold of 0.50, the model produced 378 false negatives and 2,554 false positives. Reducing the threshold to 0.40 decreased false negatives to 195, while false positives increased to 3,137.

Further reductions in the threshold continued to improve attack recall. At a threshold of 0.30, only 90 attacks were missed; however, the number of false-positive predictions increased to 3,766.

These results demonstrate the expected trade-off between attack sensitivity and false-alarm frequency. Lower thresholds favor attack detection by reducing false negatives, while slightly decreasing precision, accuracy, and F1-score.

Since this analysis was performed on the independent test dataset, the evaluated thresholds are treated as a post-hoc sensitivity analysis rather than as a basis for selecting a new operating threshold. Any operational threshold should be selected using development or validation data and subsequently evaluated on independent data.

## 19.4. Single-Observation Attack Prediction

A reusable inference function is created to demonstrate how the final model can classify an individual network observation and return its estimated attack probability.

In [72]:
def predict_attack(model, observation, feature_columns):
    input_data = pd.DataFrame(
        [observation],
        columns=feature_columns
    )

    predicted_label = model.predict(input_data)[0]

    attack_probability = model.predict_proba(
        input_data
    )[0, 1]

    predicted_class = (
        "Attack"
        if predicted_label == 1
        else "Benign"
    )

    print("Prediction:", predicted_class)
    print(
        "Attack Probability:",
        f"{attack_probability * 100:.2f}%"
    )

    return predicted_label, attack_probability

In [73]:
sample_observation = X_test.iloc[0].to_dict()

predict_attack(
    final_rf_model,
    sample_observation,
    reduced_feature_columns
)

Prediction: Attack
Attack Probability: 94.28%


(np.int64(1), np.float64(0.9428375964470356))

In [74]:
print(
    "Actual Class:",
    "Attack" if y_test.iloc[0] == 1 else "Benign"
)

Actual Class: Attack


In [75]:
benign_index = y_test[y_test == 0].index[0]

benign_observation = X_test.loc[benign_index].to_dict()

predicted_label, attack_probability = predict_attack(
    final_rf_model,
    benign_observation,
    reduced_feature_columns
)

print(
    "Actual Class:",
    "Attack" if y_test.loc[benign_index] == 1 else "Benign"
)

Prediction: Benign
Attack Probability: 11.89%
Actual Class: Benign


### 19.4. Findings

The inference examples demonstrate that the final model can classify individual network observations and provide an estimated attack probability.

For an attack observation, the model predicted the `Attack` class with an attack probability of 94.28%.

For a benign observation, the model correctly predicted the `Benign` class with an attack probability of 11.89%.

These examples illustrate how the trained model can be used in practice to provide both binary classifications and probability-based confidence estimates for individual network traffic observations.

## 19.5. Final Attack Probability Output

The final binary detection model produces an estimated attack probability for each device-level traffic instance.

For presentation purposes, representative instances are labeled as D1, D2, D3, and so on.

These probability estimates constitute the final output of the detection stage and can subsequently be used by downstream analysis components.

In [78]:
sample_indices = [0, 1, 2, 3, 4]

sample_instances = X_test.iloc[sample_indices]

sample_probabilities = final_rf_model.predict_proba(
    sample_instances
)[:, 1]

final_probability_output = pd.DataFrame({
    "Device": [
        f"D{i + 1}"
        for i in range(len(sample_instances))
    ],
    "Attack_Probability": sample_probabilities
})

display(
    final_probability_output.style.format({
        "Attack_Probability": "{:.4f}"
    })
)

,Device,Attack_Probability
0,D1,0.9428
1,D2,0.9358
2,D3,0.8544
3,D4,0.9736
4,D5,0.8284
